# Inference Optimization & Serving

Companion notebook for the [Inference Optimization lesson](https://ml-viz-ruby.vercel.app/courses/ml-in-practice/12-inference-optimization-and-serving).

**The idea in one sentence.** LLM serving is dominated by the **KV-cache** (its memory grows
with sequence length × batch and caps concurrency), by **batching strategy** (continuous
batching beats static by not waiting for the slowest request), and by the **roofline**
(decode is memory-bound, so batching raises arithmetic intensity).

We build the KV-cache math, batching simulation, and roofline from scratch, and **validate
the KV-cache scaling and roofline crossover**, then cover the gotchas.

> **To save your work:** click the **Copy to Drive** button at the top, or go to File → Save a copy in Drive.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict, deque

# Dark style matching the site theme.
plt.style.use('dark_background')
plt.rcParams.update({
    'axes.edgecolor': '#475569',
    'axes.labelcolor': '#e2e8f0',
    'xtick.color': '#94a3b8',
    'ytick.color': '#94a3b8',
    'axes.titlecolor': '#e2e8f0',
    'figure.facecolor': '#0f1117',
    'axes.facecolor': '#1a1d27',
    'grid.color': '#2e3347',
    'savefig.facecolor': '#0f1117',
})
BRAND = '#6366f1'
TEAL = '#14b8a6'
ROSE = '#f43f5e'
YELLOW = '#eab308'

rng = np.random.default_rng(0)

## 1. KV-cache memory: the thing that limits concurrency

For a decoder-only transformer with $L$ layers, $H$ KV-heads of dimension $d$, batch size $b$, sequence length $s$, and `bytes` per element, the KV-cache size is

$$\text{bytes} = 2 \cdot L \cdot H \cdot d \cdot s \cdot b \cdot \text{bytes per element}$$

We sweep across context length $s$ and batch size $b$ for a Llama-3.1-70B-shaped model and plot the resulting cache in GB. The key sanity check: at $s = 8192, b = 1$ the cache is ~2.5 GB; at $b = 32$ it's ~80 GB — a full H100. The cache, not the weights, is the binding constraint on concurrency.

In [ ]:
def kv_cache_bytes(num_layers, num_kv_heads, head_dim, seq_len, batch_size, bytes_per_elem=2):
    """KV-cache size in bytes. Factor of 2 because we store both K and V."""
    return 2 * num_layers * num_kv_heads * head_dim * seq_len * batch_size * bytes_per_elem

# Llama-3.1-70B-shaped config, FP16 weights and cache.
L, H, d = 80, 8, 128
BYTES_PER_GB = 1024 ** 3

# Spot check: s = 8192, b = 1 should be ~2.5 GB.
spot = kv_cache_bytes(L, H, d, seq_len=8192, batch_size=1) / BYTES_PER_GB
print(f'Single-sequence cache @ s=8192, b=1: {spot:.2f} GB')

# Sweep.
seq_lens = np.array([1024, 2048, 4096, 8192, 16384, 32768])
batch_sizes = [1, 4, 16, 32]

fig, ax = plt.subplots(figsize=(8, 4.8))
colors = [BRAND, TEAL, YELLOW, ROSE]
for b, c in zip(batch_sizes, colors):
    gb = kv_cache_bytes(L, H, d, seq_len=seq_lens, batch_size=b) / BYTES_PER_GB
    ax.plot(seq_lens, gb, 'o-', color=c, lw=2, markersize=7, label=f'batch size {b}')

ax.axhline(80, color='#94a3b8', linestyle='--', lw=1, alpha=0.7, label='H100 HBM (80 GB)')
ax.set_xscale('log', base=2)
ax.set_yscale('log')
ax.set_xlabel('context length s (tokens)')
ax.set_ylabel('KV-cache size (GB)')
ax.set_title('KV-cache grows linearly in s and b — quickly fills the GPU')
ax.set_xticks(seq_lens)
ax.set_xticklabels([str(s) for s in seq_lens])
ax.legend(loc='upper left', fontsize=9)
ax.grid(True, which='both', alpha=0.3)
plt.tight_layout()
plt.show()

### Validate: KV-cache memory scales linearly with sequence length and batch

The KV-cache stores K and V for every layer × head × token, so its size is linear in both
sequence length and batch — and it's what limits how many concurrent requests fit in GPU
memory. We confirm the linear scaling and the ~2.5 GB spot check.

In [ ]:
base = kv_cache_bytes(L, H, d, seq_len=8192, batch_size=1)
print(f'spot check (s=8192, b=1): {base / BYTES_PER_GB:.2f} GB')
assert abs(base / BYTES_PER_GB - 2.5) < 0.5, 'single-sequence cache is ~2.5 GB for this config'
# linear in sequence length and in batch
assert np.isclose(kv_cache_bytes(L, H, d, 16384, 1), 2 * base), 'doubling seq_len doubles the cache'
assert np.isclose(kv_cache_bytes(L, H, d, 8192, 4), 4 * base), 'quadrupling batch quadruples the cache'
print('doubling seq_len or batch scales the cache proportionally')
print('\n✅ the KV-cache grows linearly with seq_len x batch — the hard cap on concurrency')

Three takeaways visible in the plot:

- At small batch / small context the cache is negligible.
- At $b = 32, s = 8192$ the cache is ~80 GB — *the entire* H100 HBM, before model weights (~140 GB at FP16) are even loaded. In practice the model is sharded across multiple GPUs or quantized, but the structural point stands: cache scales linearly in both axes.
- Long-context serving (s = 32k+) is *only* possible with paged KV-cache (vLLM PagedAttention), KV-cache quantization (INT8/FP8), or both. A naive allocator that reserves `max_seq_len` for every active request would never fit.

## 2. Continuous vs static batching

We simulate a chat workload: 200 requests with response lengths drawn from a heavy-tailed distribution (~50–1000 tokens). We compare two policies:

- **Static batching** — gather $B = 32$ requests, run them as a batch, return responses together. The batch runs at the *longest* sequence's pace, and short sequences' slots sit idle until the longest finishes.
- **Continuous batching** — schedule one decode step at a time across the active batch. The moment any sequence finishes, a queued request takes its slot.

We measure throughput (tokens/sec) and per-request latency (p50, p95). The simulation uses 'time' in units of decode steps and assumes a fixed time per step regardless of batch size (a simplification — real GPUs see slightly higher per-step time with larger batches, but throughput-per-step still rises).

In [ ]:
def make_workload(n_requests, seed=0):
    """Mixed short/long response lengths: 70 percent short (50-200), 25 percent medium (200-500), 5 percent long (500-1000)."""
    rng_local = np.random.default_rng(seed)
    bucket = rng_local.random(n_requests)
    out = np.empty(n_requests, dtype=int)
    short_mask = bucket < 0.70
    med_mask = (bucket >= 0.70) & (bucket < 0.95)
    long_mask = bucket >= 0.95
    out[short_mask] = rng_local.integers(50, 200, size=short_mask.sum())
    out[med_mask] = rng_local.integers(200, 500, size=med_mask.sum())
    out[long_mask] = rng_local.integers(500, 1000, size=long_mask.sum())
    # Arrival times: Poisson process at 4 req per time step.
    inter = rng_local.exponential(1.0 / 4.0, size=n_requests)
    arrivals = np.cumsum(inter)
    return arrivals, out


def simulate_static_batching(arrivals, lengths, batch_size=32):
    """Gather batch_size requests, run them together for max(lengths_in_batch) steps, then start next batch."""
    n = len(arrivals)
    finish_time = np.empty(n)
    t = 0.0
    i = 0
    while i < n:
        j = min(i + batch_size, n)
        # The batch can't start before all its requests have arrived.
        start = max(t, arrivals[j - 1])
        batch_lens = lengths[i:j]
        duration = int(batch_lens.max())
        end = start + duration
        # All requests in the batch finish together (worst case for short requests).
        finish_time[i:j] = end
        t = end
        i = j
    return finish_time


def simulate_continuous_batching(arrivals, lengths, batch_size=32):
    """At every step, run the batch_size oldest waiting / in-progress requests. Slots free as requests finish."""
    n = len(arrivals)
    finish_time = np.empty(n)
    remaining = lengths.copy()
    # active is a dict mapping request idx -> remaining tokens. Step time = 1.
    active = {}
    t = 0.0
    next_arrival = 0
    while next_arrival < n or active:
        # Admit any newly-arrived requests up to batch_size.
        while next_arrival < n and arrivals[next_arrival] <= t and len(active) < batch_size:
            active[next_arrival] = remaining[next_arrival]
            next_arrival += 1
        if not active:
            # Fast-forward to the next arrival.
            t = arrivals[next_arrival]
            continue
        # One decode step: every active request decreases by 1 token.
        t += 1.0
        finished = []
        for idx in active:
            active[idx] -= 1
            if active[idx] <= 0:
                finished.append(idx)
        for idx in finished:
            finish_time[idx] = t
            del active[idx]
    return finish_time


def summarise(label, arrivals, lengths, finish_time):
    latency = finish_time - arrivals
    total_tokens = int(lengths.sum())
    makespan = float(finish_time.max() - arrivals.min())
    throughput = total_tokens / makespan
    p50 = float(np.percentile(latency, 50))
    p95 = float(np.percentile(latency, 95))
    print(f'{label:18s}  throughput = {throughput:7.1f} tok/step   '
          f'p50 latency = {p50:7.1f}   p95 latency = {p95:7.1f}   '
          f'makespan = {makespan:7.1f}')
    return {'throughput': throughput, 'p50': p50, 'p95': p95, 'makespan': makespan}


arrivals, lengths = make_workload(n_requests=200, seed=1)

static_finish = simulate_static_batching(arrivals, lengths, batch_size=32)
cont_finish = simulate_continuous_batching(arrivals, lengths, batch_size=32)

static_stats = summarise('static batch', arrivals, lengths, static_finish)
cont_stats = summarise('continuous batch', arrivals, lengths, cont_finish)

In [ ]:
static_latency = static_finish - arrivals
cont_latency = cont_finish - arrivals

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11.5, 4.5))

metrics = ['throughput', 'p50 lat', 'p95 lat']
static_vals = [static_stats['throughput'], static_stats['p50'], static_stats['p95']]
cont_vals = [cont_stats['throughput'], cont_stats['p50'], cont_stats['p95']]
x = np.arange(len(metrics))
w = 0.36
ax1.bar(x - w / 2, static_vals, w, color=ROSE, label='static batching')
ax1.bar(x + w / 2, cont_vals, w, color=TEAL, label='continuous batching')
ax1.set_xticks(x)
ax1.set_xticklabels(metrics)
ax1.set_title('Throughput up, tail latency down')
ax1.legend(loc='upper right', fontsize=9)
ax1.grid(True, alpha=0.3, axis='y')

bins = np.linspace(0, max(static_latency.max(), cont_latency.max()), 40)
ax2.hist(static_latency, bins=bins, alpha=0.7, color=ROSE, label='static')
ax2.hist(cont_latency, bins=bins, alpha=0.7, color=TEAL, label='continuous')
ax2.set_xlabel('per-request latency (decode steps)')
ax2.set_ylabel('# requests')
ax2.set_title('Static batching has a huge tail — short requests wait for the longest')
ax2.legend(loc='upper right', fontsize=9)
ax2.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

Two things to notice in the bars and the histogram:

- **Continuous batching ~doubles throughput** on this workload because static batching idles ~50% of its decode slots after the short requests finish — those slots are still 'occupied' until the longest request in the batch is done. The exact factor depends on length variance; on real chat traffic (heavier tails than this toy) the win is closer to 4–10x.
- **The static-batching latency histogram has a thick tail at the right edge.** Short requests that should have finished in 50 steps end up waiting for a 900-step neighbour to finish — they all 'land' together at the batch boundary. Continuous batching's histogram is tight against the small-end of the response-length distribution because short requests release their slot the instant they're done.

## 3. The roofline model

The roofline plot is the cleanest mental model for accelerator choice: x-axis is *operational intensity* (FLOPs per byte read from HBM), y-axis is achievable FLOPs/s. The achievable performance is the minimum of two ceilings:

$$\text{achievable FLOPs/s} = \min(\text{peak compute},\; \text{HBM bandwidth} \times \text{operational intensity})$$

Below the ridge (where the diagonal bandwidth line meets the flat compute ceiling), you're memory-bound; above it, you're compute-bound. We plot a synthetic H100 roofline and mark where LLM prefill (compute-bound) and LLM decode (memory-bound) live.

In [ ]:
PEAK_FLOPS = 1.0e15        # H100-ish: ~1 PF/s FP16
HBM_BW = 3.0e12            # ~3 TB/s
RIDGE = PEAK_FLOPS / HBM_BW  # FLOPs/byte at which the two ceilings cross

op_intensity = np.logspace(-1, 4, 200)   # 0.1 to 10_000 FLOPs/byte
memory_ceiling = HBM_BW * op_intensity
compute_ceiling = np.full_like(op_intensity, PEAK_FLOPS)
achievable = np.minimum(memory_ceiling, compute_ceiling)

# Approximate workload positions:
#   LLM decode @ b=1: ~1 FLOP/byte (single token, all weights read).
#   LLM decode @ b=32: ~32 FLOP/byte (same weight read amortised over 32 tokens).
#   LLM prefill @ s=2048: ~2000 FLOPs/byte (sequence parallelism saturates compute).
workloads = [
    ('LLM decode (b=1)',  1.0, ROSE),
    ('LLM decode (b=32)', 32.0, YELLOW),
    ('LLM prefill (s=2048)', 2000.0, TEAL),
]

fig, ax = plt.subplots(figsize=(8.5, 5))
ax.plot(op_intensity, achievable, color=BRAND, lw=2.5, label='achievable (roofline)')
ax.plot(op_intensity, memory_ceiling, color='#94a3b8', lw=1, linestyle='--', alpha=0.6, label='HBM bandwidth ceiling')
ax.plot(op_intensity, compute_ceiling, color='#94a3b8', lw=1, linestyle=':', alpha=0.6, label='peak compute ceiling')
ax.axvline(RIDGE, color='#475569', lw=1, alpha=0.6)
ax.text(RIDGE * 1.1, 5e13, f'ridge at {RIDGE:.0f} FLOPs/byte',
        color='#94a3b8', fontsize=9, rotation=0)

for label, oi, color in workloads:
    achieved = min(HBM_BW * oi, PEAK_FLOPS)
    ax.plot(oi, achieved, 'o', color=color, markersize=11, label=label)
    ax.annotate(label, (oi, achieved), xytext=(8, -6), textcoords='offset points',
                color=color, fontsize=9)

ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlim(0.1, 1e4)
ax.set_ylim(1e11, 3e15)
ax.set_xlabel('operational intensity (FLOPs / byte read from HBM)')
ax.set_ylabel('achievable FLOPs/s')
ax.set_title('Roofline model: LLM decode sits firmly in the memory-bound region')
ax.legend(loc='lower right', fontsize=9)
ax.grid(True, which='both', alpha=0.25)
plt.tight_layout()
plt.show()

print(f'Ridge point: {RIDGE:.0f} FLOPs/byte')
for label, oi, _ in workloads:
    achieved = min(HBM_BW * oi, PEAK_FLOPS)
    util = achieved / PEAK_FLOPS
    regime = 'memory-bound' if oi < RIDGE else 'compute-bound'
    print(f'  {label:24s} OI={oi:7.1f}  achieved={achieved:.2e} FLOPs/s  '
          f'({util*100:5.1f} percent of peak)  -> {regime}')

### Validate: the roofline crossover is the ridge point

Achievable performance is $\min(P_{\max}, I\cdot BW)$: below the ridge $I^*=P_{\max}/BW$ a
kernel is **memory-bound**, above it **compute-bound**. LLM decode at batch 1 has intensity
~1 (memory-bound); batching raises intensity toward the ridge. We confirm the crossover.

In [ ]:
def achievable_perf(I): return min(PEAK_FLOPS, I * HBM_BW)
print(f'ridge point I* = {RIDGE:.0f} FLOP/byte')
for I, label in [(1.0, 'decode b=1'), (32.0, 'decode b=32'), (500.0, 'prefill/GEMM')]:
    perf = achievable_perf(I)
    bound = 'memory' if I < RIDGE else 'compute'
    print(f'{label:14s} I={I:6.0f} -> {perf/1e12:6.1f} TFLOP/s [{bound}-bound]')
    if I < RIDGE:
        assert np.isclose(perf, I * HBM_BW), 'below ridge: memory-bound'
    else:
        assert np.isclose(perf, PEAK_FLOPS), 'above ridge: compute-bound'
assert 1.0 < RIDGE, 'decode at batch 1 is memory-bound (below the ridge)'
print('\n✅ decode is memory-bound; batching raises intensity toward the compute ridge')

What the plot makes obvious:

- **LLM decode at b=1 reaches ~0.3% of peak compute.** The H100's tensor cores are sitting almost idle; the GPU is reading weights from HBM as fast as it can, and that's the constraint. Adding more FLOPs (bigger chip) is wasted money on this workload.
- **Batching to b=32 brings decode to ~10% of peak compute** by amortising the same weight read across 32 tokens — operational intensity rises by ~32x, and so does decode throughput.
- **LLM prefill saturates the compute ceiling** because every input token is processed in parallel; the sequence length acts as a free amortisation of the weight read.

This is *why* a chat product's per-request cost is so sensitive to decode batch size: at b=1 you pay for the whole GPU and use a few percent of it; batching is what makes inference economics work at all.

## Gotchas & tradeoffs

| Gotcha | Consequence |
|--------|-------------|
| **KV-cache memory** | grows with seq_len x batch (verified) — the hard cap on concurrency |
| **static batching** | waits for the slowest request in the batch; continuous batching avoids it |
| **decode is memory-bound** | adding compute doesn't help; batch to raise intensity (verified) |
| **long contexts** | the cache dominates memory; use paged attention / GQA / cache quantization |
| **latency vs throughput** | bigger batches raise throughput but each request waits longer |

Demo: batching raises arithmetic intensity toward the compute ridge.

In [ ]:
# Why batching is the key serving lever: it amortises the (memory-bound) weight read over
# many sequences, raising arithmetic intensity toward the compute ridge — more throughput
# for the same weight-loading cost. We show intensity rising with batch size.
for b in [1, 8, 32, 128]:
    # decode intensity ~ batch (weight bytes read once, reused across the batch)
    intensity = float(b)
    bound = 'memory' if intensity < RIDGE else 'compute'
    print(f'batch {b:3d}: arithmetic intensity ~ {intensity:.0f} FLOP/byte [{bound}-bound]')
print(f'\nridge is at {RIDGE:.0f} FLOP/byte -> you need a large batch to become compute-bound.')
print('Batching trades per-request latency for throughput by amortising the weight read.')

## Your turn — predict the KV-cache memory of a deployment

Your goal: implement `cache_size_gb(num_layers, num_kv_heads, head_dim, seq_len, batch_size, bytes_per_elem)` that returns the KV-cache size in GiB ($1024^3$ bytes).

Then assert two boundary checks:

- Llama-3.1-70B-shaped ($L=80, H=8, d=128$), FP16, $s=8192, b=1$ — should be ~2.5 GiB.
- Same model at $b=32$ — should be ~80 GiB (about a full H100).

In [ ]:
def cache_size_gb(num_layers, num_kv_heads, head_dim, seq_len, batch_size, bytes_per_elem=2):
    """KV-cache size in GiB.

    TODO(you):
    1. Compute total bytes: 2 * num_layers * num_kv_heads * head_dim * seq_len * batch_size * bytes_per_elem.
    2. Convert to GiB by dividing by 1024**3.
    """
    # TODO
    return ...

In [ ]:
one_seq = cache_size_gb(80, 8, 128, 8192, 1, 2)
batch_32 = cache_size_gb(80, 8, 128, 8192, 32, 2)

print(f'Single sequence cache: {one_seq:.2f} GiB  (expected ~2.5)')
print(f'Batch of 32 cache:     {batch_32:.2f} GiB  (expected ~80.0)')

assert abs(one_seq - 2.5) < 0.05, f'one_seq should be ~2.5 GiB, got {one_seq}'
assert abs(batch_32 - 80.0) < 1.0, f'batch_32 should be ~80 GiB, got {batch_32}'
assert abs(batch_32 / one_seq - 32) < 0.1, 'cache should scale linearly with batch'
print('\nAll asserts pass. The cache, not the weights, sets the concurrency limit.')

<details>
<summary>Solution</summary>

```python
def cache_size_gb(num_layers, num_kv_heads, head_dim, seq_len, batch_size, bytes_per_elem=2):
    total_bytes = 2 * num_layers * num_kv_heads * head_dim * seq_len * batch_size * bytes_per_elem
    return total_bytes / (1024 ** 3)
```

Two operational implications worth carrying around:

- *The cache scales linearly in both context length and batch size.* A 4x context-window upgrade or a 4x concurrency upgrade has the same memory cost. PagedAttention's wins (60–80% HBM recovered) come from not pre-reserving the worst case in either axis.
- *KV-cache quantization (INT8 or FP8) halves these numbers* with negligible quality loss. Combined with paged allocation, this is what unlocks 32k–128k context windows on a single GPU.

</details>

## Recap

- The **KV-cache** is what gets the GPU full long before the model weights do. Plan capacity around the cache.
- **Continuous batching** beats static batching by 4–10x on chat traffic because the batch never waits for the slowest sequence — slots free the instant any sequence finishes.
- The **roofline model** tells you, in one picture, whether your workload wants bandwidth (LLM decode) or compute (LLM prefill, CNNs). Buy accordingly.
- In 2026, the right *default* answer for self-hosted LLM serving is vLLM (PagedAttention + continuous batching), then add quantization (INT4 weights + FP8 KV-cache) and speculative decoding once you've measured the baseline.